# Calculus singing engine — Colab GPU
Run the numbered cells in order. First choose **Runtime → Change runtime type → T4 GPU**.
This runs ACE-Step, a pretrained **lyrics-to-song** model. It does not train a new model.
Generation makes vocals and accompaniment. A song title alone does not encode its melody;
without a supplied melody reference, the output is an original song in the requested style.

The first setup downloads dependencies and several GB of model weights. Keep this notebook
connected during the Android demo. This notebook has been prepared and syntax-checked;
its execution on your Colab GPU must still be verified.


In [ ]:
# 1. Install the official runtime in an isolated Python 3.12 environment.
import os, sys, subprocess, pathlib
subprocess.run(["nvidia-smi"], check=True)
ROOT = pathlib.Path("/content/ACE-Step-1.5")
if not ROOT.exists():
    subprocess.check_call(["git", "clone", "https://github.com/ace-step/ACE-Step-1.5.git", str(ROOT)])
subprocess.check_call(["git", "checkout", "ca1e85fe9430179831e6bc6be790c332190a3866"], cwd=ROOT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
subprocess.check_call(["uv", "sync", "--frozen", "--python", "3.12"], cwd=ROOT)
print("Dependencies installed. Run cell 2.")


In [ ]:
# 2. Start the singing model. Wait for MODEL READY before continuing.
import secrets, time, requests
# The pinned loader does not yet implement the ACESTEP_DTYPE override suggested by its own NaN error.
# Add a scoped, opt-in override before loading weights. Default behavior is unchanged without the env flag.
loader = ROOT / "acestep/core/generation/handler/init_service_orchestrator.py"
loader_source = loader.read_text()
precision_marker = "# Song Studio precision override"
anchor = "            self.quantization = normalized_quantization"
if precision_marker not in loader_source:
    assert loader_source.count(anchor) == 1, "Loader version changed; inspect the precision patch before applying."
    override = ('            # Song Studio precision override\n'
                '            if os.environ.get("ACESTEP_DTYPE", "").strip().lower() == "float32":\n'
                '                self.dtype = torch.float32\n')
    loader_source = loader_source.replace(anchor, override + anchor)
    compile(loader_source, str(loader), "exec")
    loader.write_text(loader_source)
previous_engine = globals().get("engine")
if previous_engine is not None and previous_engine.poll() is None:
    previous_engine.terminate()
    try:
        previous_engine.wait(timeout=30)
    except subprocess.TimeoutExpired:
        previous_engine.kill()
        previous_engine.wait()
API_KEY = secrets.token_urlsafe(32)
env = os.environ.copy()
env.update(ACESTEP_INIT_LLM="false", ACESTEP_DEVICE="cuda", ACESTEP_DTYPE="float32",
           ACESTEP_OFFLOAD_TO_CPU="true", ACESTEP_USE_FLASH_ATTENTION="false",
           ACESTEP_CONFIG_PATH="acestep-v15-turbo", ACESTEP_API_KEY=API_KEY,
           PYTHONUNBUFFERED="1", MPLBACKEND="Agg")
LOG_PATH = pathlib.Path("/content/ace-server.log")
log_file = LOG_PATH.open("w")
engine = subprocess.Popen([str(ROOT / ".venv/bin/acestep-api"), "--host", "127.0.0.1", "--port", "8001"],
                          cwd=ROOT, env=env, stdout=log_file, stderr=subprocess.STDOUT)
for attempt in range(600):
    if engine.poll() is not None:
        print(LOG_PATH.read_text()[-6000:])
        raise RuntimeError("The model stopped. Share the error text, not API keys.")
    try:
        r = requests.get("http://127.0.0.1:8001/health", timeout=3,
                         headers={"Authorization": "Bearer " + API_KEY})
        if r.ok:
            print("MODEL READY. Run cell 3 to test singing.")
            break
    except requests.RequestException:
        pass
    if attempt % 15 == 0:
        print("Loading model / downloading weights…", attempt * 2, "seconds")
    time.sleep(2)
else:
    print(LOG_PATH.read_text()[-6000:])
    raise TimeoutError("Model startup exceeded 20 minutes.")


In [ ]:
# 3. Generate and listen to a real 30-second Calculus song.
from IPython.display import Audio, display
import json
headers = {"Authorization": "Bearer " + API_KEY}
lyrics = """[verse]
Continuous from A to B
Differentiable inside
Equal values at the ends
Let the tangent be our guide
[chorus]
Somewhere C lies in between
There the slope is zero
Rolle gives a point unseen
Flat tangent is the hero"""
payload = dict(prompt="English educational pop song, bright synths, clear warm singing, rhythmic bass, memorable chorus",
               lyrics=lyrics, vocal_language="en", bpm=100, audio_duration=30,
               inference_steps=8, batch_size=1, thinking=False, use_cot_caption=False,
               use_cot_language=False, key_scale="C Major", time_signature="4", audio_format="mp3")
r = requests.post("http://127.0.0.1:8001/release_task", json=payload, headers=headers, timeout=30)
r.raise_for_status()
task = r.json()["data"]["task_id"]
for attempt in range(360):
    r = requests.post("http://127.0.0.1:8001/query_result", json={"task_id_list": [task]}, headers=headers, timeout=30)
    r.raise_for_status()
    rows = r.json()["data"]
    if rows and rows[0]["status"] == 2:
        server_log = pathlib.Path("/content/ace-server.log").read_text(errors="replace")
        print(server_log.replace(API_KEY, "[REDACTED]")[-16000:])
        raise RuntimeError("Generation failed. The server traceback is printed above; share that traceback.")
    if rows and rows[0]["status"] == 1:
        result = json.loads(rows[0]["result"])[0]
        from urllib.parse import urlparse
        audio_url = urlparse(result["file"])
        audio = requests.get("http://127.0.0.1:8001" + audio_url.path + "?" + audio_url.query,
                             headers=headers, timeout=60)
        audio.raise_for_status()
        pathlib.Path("/content/calculus-demo.mp3").write_bytes(audio.content)
        display(Audio(audio.content))
        print("Singing test complete. Listen, then run cell 4.")
        break
    if attempt % 10 == 0:
        print("Generating vocals and music…")
    time.sleep(2)
else:
    raise TimeoutError("Generation exceeded 12 minutes. Inspect /content/ace-server.log.")


In [ ]:
# 4. Connect this temporary, API-key-protected service to your Mac.
import re, urllib.request
from google.colab import files
# Confirm the existing authenticated engine before publishing a new tunnel.
if "API_KEY" not in globals():
    raise RuntimeError("Run cell 2 first and wait for MODEL READY, then run this cell.")
try:
    ready = requests.post("http://127.0.0.1:8001/query_result", json={"task_id_list": []},
                          headers={"Authorization": "Bearer " + API_KEY}, timeout=15)
    ready.raise_for_status()
    assert ready.json().get("code") == 200
except Exception:
    raise RuntimeError("The model API is not ready. Run cell 2, wait for MODEL READY, then rerun cell 4.") from None
previous_tunnel = globals().get("tunnel")
if previous_tunnel is not None and previous_tunnel.poll() is None:
    previous_tunnel.terminate()
    try:
        previous_tunnel.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous_tunnel.kill()
        previous_tunnel.wait()
binary = pathlib.Path("/content/cloudflared")
if not binary.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", str(binary))
    binary.chmod(0o755)
tunnel_log = pathlib.Path("/content/music-tunnel.log")
tunnel = subprocess.Popen([str(binary), "tunnel", "--url", "http://127.0.0.1:8001"],
                          stdout=tunnel_log.open("w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    text = tunnel_log.read_text() if tunnel_log.exists() else ""
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        url = match.group(0)
        break
    time.sleep(1)
if not url:
    raise RuntimeError("Tunnel could not start. Inspect /content/music-tunnel.log.")
config = pathlib.Path("/content/song-service.env")
config.write_text("LOCAL_MUSIC_URL=" + url + "\nLOCAL_MUSIC_API_KEY=" + API_KEY + "\n")
print("Service URL:", url)
print("Downloading song-service.env. Move it into app_de_music/server and rename it .env.")
print("Do NOT paste the config file contents or API key into chat or GitHub.")
files.download(str(config))
